# 위험구역 침입 감지 — 학습 및 성능 평가

데이터셋 두 가지(`merged` / `merged_team`)를 **같은 조건**으로 학습하고
성능 지표와 그래프를 모두 뽑는다.

> **먼저** : 메뉴 `런타임 → 런타임 유형 변경 → T4 GPU`  (CPU면 10분짜리가 2시간)

**클래스** : `0 baby` · `1 adult` · `2 knife` · `3 outlet`


## 0. 지표 읽는 법 — 결과 보기 전에

| 지표 | 뜻 | 낮으면 |
|---|---|---|
| **Precision (정밀도)** | 있다고 한 것 중 진짜 비율 | **오검출**이 많다 |
| **Recall (재현율)** | 실제 있는 것 중 찾아낸 비율 | **놓친다** |
| **mAP50** | 박스가 대충 맞으면 정답 | 아예 못 찾는다 |
| **mAP50-95** | 박스 정확도까지 엄격히 채점 | 찾긴 하는데 박스가 어긋난다 |

우리 시나리오는 **아기를 놓치면 사고**이므로 **Recall이 더 중요하다.**
보고서에 "mAP만" 쓰지 말고 Precision/Recall을 같이 쓸 것.


## 1. 환경


In [ ]:
!pip -q install ultralytics
import torch, ultralytics, os, json, glob, collections
print('ultralytics', ultralytics.__version__)
print('GPU :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else '없음 — 런타임을 T4로!')


## 2. 데이터 올리기

파일 선택 창에서 **두 개를 한 번에** 고르세요 (Ctrl 누르고 클릭).

```
C:\Users\kccistc\Developer\침입감지_프로젝트\merged.zip
C:\Users\kccistc\Developer\침입감지_프로젝트\merged_team.zip
```

하나만 돌릴 거면 하나만 고르고, 아래 `DATASETS` 에서 나머지 줄을 지우면 된다.


In [ ]:
from google.colab import files
up = files.upload()
for z in up:
    !unzip -q -o "{z}" -d /content/
!ls /content


In [ ]:
DATASETS = {
    'mine': '/content/merged',
    'team': '/content/merged_team',
}
DATASETS = {k: v for k, v in DATASETS.items() if os.path.isdir(v)}

for tag, root in DATASETS.items():
    y = os.path.join(root, 'data.yaml')
    txt = open(y, encoding='utf-8').read()
    keep = [l for l in txt.splitlines() if not l.startswith('path:')]
    open(y, 'w', encoding='utf-8').write('path: %s\n' % root + '\n'.join(keep))
print('사용할 데이터셋 :', list(DATASETS))


## 3. 데이터 점검 — 학습 전에 반드시
클래스별 박스 수가 0이면 그 클래스는 **아예 못 배운다.**


In [ ]:
CLASSES = ['baby', 'adult', 'knife', 'outlet']

def stat(root):
    out = {}
    for sp in ['train', 'val', 'test']:
        di, dl = os.path.join(root, 'images', sp), os.path.join(root, 'labels', sp)
        if not os.path.isdir(di):
            continue
        c, nl = collections.Counter(), 0
        for f in os.listdir(dl):
            if f.endswith('.txt') and f != 'classes.txt':
                nl += 1
                for line in open(os.path.join(dl, f)):
                    if line.strip():
                        c[int(line.split()[0])] += 1
        out[sp] = (len(os.listdir(di)), nl, c)
    return out

for tag, root in DATASETS.items():
    print('[%s]' % tag)
    for sp, (ni, nl, c) in stat(root).items():
        s = ', '.join('%s %d' % (CLASSES[k], v) for k, v in sorted(c.items()))
        print('   %-5s 사진 %4d / 라벨 %4d   %s' % (sp, ni, nl, s))
    print()


## 4. 학습

`seed=0` 을 고정한다. 안 하면 같은 데이터로도 결과가 달라져서
**차이가 데이터 때문인지 운 때문인지 구분할 수 없다.**


In [ ]:
from ultralytics import YOLO

COMMON = dict(epochs=100, imgsz=640, batch=16, patience=30,
              fliplr=0.5, degrees=10.0, seed=0, deterministic=True,
              project='runs', exist_ok=True, plots=True, verbose=False)

result = {}
for tag, root in DATASETS.items():
    print('=' * 62)
    print(' 학습 :', tag)
    print('=' * 62)
    m = YOLO('yolo11n.pt')
    m.train(data=os.path.join(root, 'data.yaml'), name=tag, **COMMON)
    v = m.val(plots=True)
    pc = {}
    for i, ci in enumerate(v.box.ap_class_index):
        pc[CLASSES[int(ci)]] = dict(
            P=float(v.box.p[i]), R=float(v.box.r[i]),
            mAP50=float(v.box.ap50[i]), mAP=float(v.box.ap[i]))
    result[tag] = dict(
        P=float(v.box.mp), R=float(v.box.mr),
        mAP50=float(v.box.map50), mAP=float(v.box.map),
        speed=dict(v.speed), per_class=pc,
        weights='runs/%s/weights/best.pt' % tag)
    print('%s 완료  P %.3f  R %.3f  mAP50 %.4f  mAP50-95 %.4f'
          % (tag, v.box.mp, v.box.mr, v.box.map50, v.box.map))


## 5. 성능 요약표
**이 표를 그대로 보고서에 넣는다.**


In [ ]:
print('%-8s %8s %8s %10s %12s %10s' % ('데이터셋','정밀도','재현율','mAP50','mAP50-95','추론(ms)'))
print('-' * 60)
for t, r in result.items():
    print('%-8s %8.3f %8.3f %10.4f %12.4f %10.1f'
          % (t, r['P'], r['R'], r['mAP50'], r['mAP'], r['speed'].get('inference', 0)))

print()
print('클래스별 (mAP50 / 정밀도 / 재현율)')
for t, r in result.items():
    print(' [%s]' % t)
    for c in CLASSES:
        d = r['per_class'].get(c)
        if d is None:
            print('   %-8s  검증 데이터에 없음' % c)
        else:
            print('   %-8s  mAP50 %.4f   P %.3f   R %.3f' % (c, d['mAP50'], d['P'], d['R']))

json.dump(result, open('/content/compare.json','w'), indent=2, ensure_ascii=False)


## 6. 그래프 — Ultralytics가 자동으로 만든 것

| 파일 | 무엇을 보는가 |
|---|---|
| `results.png` | **학습 곡선.** loss가 내려가고 mAP가 올라가는지. 끝에서 평평하면 충분히 학습된 것 |
| `confusion_matrix_normalized.png` | **무엇을 무엇으로 착각하는지.** 대각선이 진할수록 좋다 |
| `PR_curve.png` | 정밀도-재현율 곡선. 곡선 아래 넓이가 mAP |
| `F1_curve.png` | **판정 임계값을 얼마로 둘지** 고를 때 본다. 꼭대기 지점이 최적 |
| `P_curve.png` / `R_curve.png` | 임계값에 따른 정밀도 / 재현율 |
| `labels.jpg` | 박스 크기·위치 분포. 데이터가 한쪽에 쏠렸는지 |
| `val_batch0_pred.jpg` | **실제 예측 결과 사진.** 눈으로 확인하는 것이 제일 확실하다 |


In [ ]:
from IPython.display import Image, display, Markdown
PLOTS = ['results.png', 'confusion_matrix_normalized.png', 'confusion_matrix.png',
         'PR_curve.png', 'F1_curve.png', 'P_curve.png', 'R_curve.png',
         'labels.jpg', 'val_batch0_labels.jpg', 'val_batch0_pred.jpg']
for tag in DATASETS:
    display(Markdown('## ▶ %s' % tag))
    for p in PLOTS:
        for f in glob.glob('runs/%s/**/%s' % (tag, p), recursive=True)[:1]:
            display(Markdown('**%s**' % p))
            display(Image(filename=f, width=780))


## 7. 직접 그리는 비교 그래프
여러 데이터셋을 나란히 놓고 보는 그림은 Ultralytics가 안 만들어 주므로 직접 그린다.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
plt.rcParams['axes.unicode_minus'] = False

tags = list(result)

# (1) 전체 지표 비교
metrics = ['P', 'R', 'mAP50', 'mAP']
labels_m = ['Precision', 'Recall', 'mAP50', 'mAP50-95']
x = np.arange(len(metrics)); w = 0.8 / max(1, len(tags))
fig, ax = plt.subplots(figsize=(8, 4.5))
for i, t in enumerate(tags):
    vals = [result[t][m] for m in metrics]
    b = ax.bar(x + i * w - 0.4 + w / 2, vals, w, label=t)
    ax.bar_label(b, fmt='%.3f', fontsize=8)
ax.set_xticks(x); ax.set_xticklabels(labels_m)
ax.set_ylim(0, 1.05); ax.set_ylabel('score'); ax.legend()
ax.set_title('Overall metrics')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.savefig('/content/cmp_overall.png', dpi=140); plt.show()

# (2) 클래스별 mAP50
present = [c for c in CLASSES if any(c in result[t]['per_class'] for t in tags)]
x = np.arange(len(present))
fig, ax = plt.subplots(figsize=(8, 4.5))
for i, t in enumerate(tags):
    vals = [result[t]['per_class'].get(c, {}).get('mAP50', 0) for c in present]
    b = ax.bar(x + i * w - 0.4 + w / 2, vals, w, label=t)
    ax.bar_label(b, fmt='%.3f', fontsize=8)
ax.set_xticks(x); ax.set_xticklabels(present)
ax.set_ylim(0, 1.05); ax.set_ylabel('mAP50'); ax.legend()
ax.set_title('Per-class mAP50')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.savefig('/content/cmp_class.png', dpi=140); plt.show()


## 8. 눈으로 확인 — 실제 예측 그려 보기
숫자만 보면 안 된다. **어디서 틀리는지**는 사진을 봐야 안다.


In [ ]:
best_tag = max(result, key=lambda t: result[t]['mAP50'])
print('더 좋은 쪽 :', best_tag)
mb = YOLO(result[best_tag]['weights'])

root = DATASETS[best_tag]
vd = os.path.join(root, 'images', 'val')
if not os.path.isdir(vd):
    vd = os.path.join(root, 'images', 'test')
imgs = sorted(glob.glob(vd + '/*.jpg'))[:8]
res = mb.predict(imgs, conf=0.25, save=True, project='runs', name='pred', exist_ok=True)
for f in sorted(glob.glob('runs/pred/*.jpg'))[:8]:
    display(Image(filename=f, width=520))


## 9. tflite 변환 (보드용)


In [ ]:
m2 = YOLO(result[best_tag]['weights'])
p32 = m2.export(format='tflite', imgsz=320)
p8 = m2.export(format='tflite', imgsz=320, int8=True,
               data=os.path.join(DATASETS[best_tag], 'data.yaml'))
for f in sorted(glob.glob('runs/%s/weights/**/*.tflite' % best_tag, recursive=True)):
    print('%8.2f MB  %s' % (os.path.getsize(f)/1048576.0, f))


## 10. 전부 내려받기
그래프·수치·모델을 한 번에 zip으로 받는다. **이걸 팀에 공유하면 된다.**


In [ ]:
import shutil
os.makedirs('/content/out', exist_ok=True)
shutil.copy(result[best_tag]['weights'], '/content/out/best.pt')
for f in glob.glob('runs/%s/weights/**/*.tflite' % best_tag, recursive=True):
    shutil.copy(f, '/content/out/' + os.path.basename(f))
for tag in DATASETS:
    for p in PLOTS:
        for f in glob.glob('runs/%s/**/%s' % (tag, p), recursive=True)[:1]:
            shutil.copy(f, '/content/out/%s_%s' % (tag, os.path.basename(f)))
    src = 'runs/%s/results.csv' % tag
    if os.path.exists(src):
        shutil.copy(src, '/content/out/%s_results.csv' % tag)
for f in ['/content/cmp_overall.png', '/content/cmp_class.png', '/content/compare.json']:
    if os.path.exists(f):
        shutil.copy(f, '/content/out/')
print('담은 파일 %d개' % len(os.listdir('/content/out')))
!cd /content && zip -qr result.zip out
files.download('/content/result.zip')


---
## 보고서에 넣을 것

1. **성능 요약표** (5번) — Precision / Recall / mAP50 / mAP50-95, 그리고 **조건**(입력 640, epoch 100, 데이터 588장)
2. **학습 곡선** `results.png` — 충분히 학습됐는지
3. **혼동행렬** — 무엇을 무엇으로 착각하는지
4. **F1 곡선** — 판정 임계값을 이 그래프 보고 정했다고 쓸 수 있다
5. **예측 사진** — 성공 사례와 **실패 사례를 같이**

> 수치 옆에 **조건을 반드시 적는다.** "mAP50 0.95" 만 쓰면 무슨 조건인지 알 수 없다.

## 보드로 옮기기
```
scp best_int8*.tflite willtek@10.10.15.61:/home/willtek/work/baby/
```
